## Deep BSDE Solver for a European Call Option with Black-Scholes dynamics

This notebook illustrates the use of a Deep BSDE solver from [E, Han and Jentzen (2017)](https://doi.org/10.1007/s40304-017-0117-6) and [Han, Jentzen and E (2018)](https://www.pnas.org/doi/abs/10.1073/pnas.1718942115) to price a European call option using the Black-Scholes model in BSDE form. 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import math
import time
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

### Payoff and Network

In [ ]:
# European call payoff function
def payoff(S, K):
    return torch.clamp(S - K, min=0.0)

In [ ]:
# A simple fully connected network for approximating Z at each time step
class FCNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

### Pricing Function
The BSDE takes the form
$$d Y_t=-f\left(t, S_t, Y_t, Z_t\right) d t+Z_t d W_t, \quad Y_T=\left(S_T-K\right)^{+}$$
where $f(t, S, Y, Z)=-r Y$.

In [ ]:
def BSDE_BlackScholes(S0, K, sigma, r, T, M=256, N=50, hidden_dim=128, max_iterations=10000, tol=1e-7, seed=42, lr=1e-2):
    start_time = time.time()
    dt = T / N
    torch.manual_seed(seed)
    
    # Build a network for each time step
    Z_nets = nn.ModuleList([FCNet(1, hidden_dim, 1).to(device) for _ in range(N - 1)])
    
    # Y0 is the initial condition (to be learned) for the BSDE
    initial_guess = S0 * np.exp(r * T) + 0.5 * sigma * np.sqrt(T) * S0 - K
    Y0 = torch.tensor([initial_guess], device=device, requires_grad=True)
    
    # Combine all trainable parameters (Y0 and the networks' parameters)
    optimizer = optim.Adam([Y0] + list(Z_nets.parameters()), lr=lr)
    
    loss_lst = []
    prev_Y0 = None
    iteration = 0
    
    for it in tqdm(range(max_iterations)):
        iteration += 1
        # Initialize S and Y for all sample paths
        S = torch.full((M, 1), S0, device=device)
        Y = Y0.repeat(M, 1)
    
        # Pre-sample Brownian increments (for each time step)
        dW = torch.randn(M, N - 1, device=device) * torch.sqrt(torch.tensor(dt))
    
        # Forward simulation along the discretized time grid
        for n in range(N - 1):
            # Compute the control process Z using the nth network given the current state S
            Z = Z_nets[n](S)
            
            # BSDE generator: for Black-Scholes, we take f = -r * Y so that the update becomes:
            # Y_{n+1} = Y_n + r*Y_n*dt + Z*dW
            f = -r * Y
            Y = Y - f * dt + Z * dW[:, n:n+1]
    
            # Update the underlying asset S using an exponential Euler scheme
            S = S * torch.exp((r - 0.5 * sigma ** 2) * dt + sigma * dW[:, n:n+1])
            
        # Compute the loss: mean squared error between Y (approximate solution at T) and the terminal payoff
        loss = torch.mean((Y - payoff(S, K))**2)
        loss_lst.append(loss.item())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Check convergence on Y0 (the BSDE solution at time 0)
        current_Y0 = Y0.item()
        if prev_Y0 is not None:
            rel_change = abs(current_Y0 - prev_Y0) / abs(prev_Y0)
            if rel_change < tol:
                print(f"Converged at iteration {iteration} with relative change in Y0: {rel_change:.2e}")
                break
        prev_Y0 = current_Y0

    end_time = time.time()
    execution_time = end_time - start_time 
    return Y0.item(), loss_lst, iteration, execution_time

In [ ]:
T = 1.0            # Maturity
r = 0.05           # Risk-free rate
sigma = 0.2        # Volatility
S0 = 100.0         # Initial stock price
K = 100.0          # Strike price

In [ ]:
Y0, loss_lst, iterations, etime = BSDE_BlackScholes(S0, K, sigma, r, T)

In [ ]:
Y0

In [ ]:
etime

In [ ]:
def plot_losses(losses, filename):
    epochs = np.arange(len(losses)) + 1
    plt.figure(figsize=(15, 10))
    plt.plot(epochs, losses,  color='black',  label='BSDE Training Loss')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_losses(loss_lst, 'bsdeloss.png')

### Sensitivity to N

In [ ]:
NList = [10, 20, 50, 100, 200]

In [ ]:
resdict = dict()
for n in NList:
    Y0, _, _, etime = BSDE_BlackScholes(S0, K, sigma, r, T, N=n)
    resdict[n] = {'value': Y0, 'etime': etime}

In [ ]:
nlistres = pd.DataFrame(resdict)

In [ ]:
nlistres

In [ ]:
nlistres.to_latex()

### Sensitivity to Hidden Size

In [ ]:
widthList = [32, 64, 128, 256, 512]

In [ ]:
resdict = dict()
for n in widthList:
    Y0, _, _, etime = BSDE_BlackScholes(S0, K, sigma, r, T, hidden_dim=n)
    resdict[n] = {'value': Y0, 'etime': etime}

In [ ]:
wlistres = pd.DataFrame(resdict)

In [ ]:
wlistres

In [ ]:
wlistres.to_latex()

### Different Learning Rates

In [ ]:
lrlist = [1e-1, 3e-2, 1e-2, 3e-3, 1e-3]

In [ ]:
resdict = dict()
for n in lrlist:
    Y0, _, _, etime = BSDE_BlackScholes(S0, K, sigma, r, T, lr=n)
    resdict[n] = {'value': Y0, 'etime': etime}

In [ ]:
lrlistres = pd.DataFrame(resdict)

In [ ]:
lrlistres

In [ ]:
lrlistres.to_latex()

### Batch Size

In [ ]:
batchsizelist = [64, 256, 1024]

In [ ]:
resdict = dict()
for n in batchsizelist:
    Y0, _, _, etime = BSDE_BlackScholes(S0, K, sigma, r, T, M=n)
    resdict[n] = {'value': Y0, 'etime': etime}

In [ ]:
batchlistres = pd.DataFrame(resdict)

In [ ]:
batchlistres

In [ ]:
batchlistres.to_latex()

### Test the Model

In [ ]:
def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / np.sqrt(2.0)))

def black_scholes(S0, K, sigma, r, T, option_type='call'):

    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        return (S0 * norm_cdf(d1) - K * np.exp(-r * T) * norm_cdf(d2)).item()
    elif option_type == 'put':
        return (K * np.exp(-r * T) * norm_cdf(-d2) - S0 * norm_cdf(-d1)).item()
    else:
        raise ValueError("option_type must be either 'call' or 'put'")

In [ ]:
black_scholes(S0, K, sigma, r, T)

In [ ]:
# Strike range and maturities to test
Kmin = 80
Kmax = 120
n_strikes = 5  # number of strike prices between Kmin and Kmax
strike_values = np.linspace(Kmin, Kmax, n_strikes)

T_values = [0.25, 0.5, 1.0]  # maturities

In [ ]:
# Store results in a list of dictionaries
results = []

for T in T_values:
    for K in strike_values:
        print(f"\nRunning BSDE for maturity T = {T}, strike K = {K}")
        Y0_bsde, loss_lst, iterations, exec_time = BSDE_BlackScholes(S0, K, sigma, r, T, M=1024, N=200, lr=3e-2)
        bs_price = black_scholes(S0, K, sigma, r, T, option_type='call')
        results.append({
            "Maturity": T,
            "Strike": K,
            "Y0_BSDE": Y0_bsde,
            "Iterations": iterations,
            "Execution_Time_sec": exec_time,
            "Black_Scholes_Price": bs_price
        })

# Convert results to a Pandas DataFrame and display
df_results = pd.DataFrame(results)

In [ ]:
df_results

In [ ]:
df_results.to_latex()